In [ ]:
!pip install pymupdf pillow

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import fitz

pdf_path = "2015개정경제수학-광주교육청.pdf"
doc = fitz.open(pdf_path)

print("PDF 파일명:", pdf_path)
print("전체 페이지 수:", len(doc))

In [ ]:
import os
import fitz

pdf_path = "2015개정경제수학-광주교육청.pdf"
doc = fitz.open(pdf_path)

output_dir = "outputs/pages"
os.makedirs(output_dir, exist_ok=True)

for page_index in range(len(doc)):
    page = doc[page_index]

    # OCR 성능을 위해 2배 해상도로 변환
    matrix = fitz.Matrix(2, 2)
    pix = page.get_pixmap(matrix=matrix)

    page_number = page_index + 1
    image_path = f"{output_dir}/page_{page_number:03}.png"

    pix.save(image_path)

    print(f"{page_number}/{len(doc)} 페이지 저장 완료: {image_path}")

print("전체 페이지 이미지 변환 완료")

In [ ]:
import os

page_files = sorted(os.listdir("outputs/pages"))

print("저장된 페이지 이미지 개수:", len(page_files))
print("처음 10개:", page_files[:10])
print("마지막 10개:", page_files[-10:])

In [ ]:
import os
from PIL import Image

input_dir = "outputs/pages"
preprocess_dir = "outputs/preprocessed_pages"
os.makedirs(preprocess_dir, exist_ok=True)

for file_name in sorted(os.listdir(input_dir)):
    if file_name.endswith(".png"):
        image_path = os.path.join(input_dir, file_name)
        img = Image.open(image_path)

        # 그레이스케일 변환
        gray_img = img.convert("L")

        save_path = os.path.join(preprocess_dir, file_name)
        gray_img.save(save_path)

        print("전처리 저장 완료:", save_path)

print("전체 페이지 전처리 완료")

In [ ]:
import os

page_files = os.listdir("outputs/pages")
preprocessed_files = os.listdir("outputs/preprocessed_pages")

print("원본 페이지 이미지 개수:", len(page_files))
print("전처리 페이지 이미지 개수:", len(preprocessed_files))

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

check_pages = [1, 8, 11, 16, 17]

for page_number in check_pages:
    image_path = f"outputs/pages/page_{page_number:03}.png"
    img = Image.open(image_path)

    plt.figure(figsize=(8, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"Page {page_number}")
    plt.show()

In [ ]:
!apt-get update
!apt-get install -y tesseract-ocr
!apt-get install -y tesseract-ocr-kor
!apt-get install -y tesseract-ocr-eng
!pip install pytesseract

In [ ]:
import pytesseract
from PIL import Image
import os

print("Tesseract OCR 준비 완료")

In [ ]:
import os

input_dir = "outputs/pages"
ocr_output_dir = "outputs/ocr_results"

os.makedirs(ocr_output_dir, exist_ok=True)

print("OCR 입력 폴더:", input_dir)
print("OCR 결과 저장 폴더:", ocr_output_dir)

In [ ]:
test_image_path = "outputs/pages/page_008.png"

img = Image.open(test_image_path)

text = pytesseract.image_to_string(
    img,
    lang="kor+eng",
    config="--psm 6"
)

print(text[:1000])

In [ ]:
image_files = sorted([
    file for file in os.listdir(input_dir)
    if file.endswith(".png")
])

print("OCR 대상 이미지 수:", len(image_files))

for idx, file_name in enumerate(image_files):
    image_path = os.path.join(input_dir, file_name)

    img = Image.open(image_path)

    page_text = pytesseract.image_to_string(
        img,
        lang="kor+eng",
        config="--psm 6"
    )

    txt_file_name = file_name.replace(".png", ".txt")
    txt_path = os.path.join(ocr_output_dir, txt_file_name)

    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(page_text)

    print(f"{idx + 1}/{len(image_files)} OCR 완료:", txt_path)

print("전체 OCR 완료")

In [ ]:
import re

cleaned_output_dir = "outputs/cleaned_text"
os.makedirs(cleaned_output_dir, exist_ok=True)

def clean_ocr_text(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = text.strip()
    return text

for file_name in sorted(os.listdir(ocr_output_dir)):
    if file_name.endswith(".txt"):
        txt_path = os.path.join(ocr_output_dir, file_name)

        with open(txt_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

        cleaned_text = clean_ocr_text(raw_text)

        save_path = os.path.join(cleaned_output_dir, file_name)

        with open(save_path, "w", encoding="utf-8") as f:
            f.write(cleaned_text)

        print("정제 완료:", save_path)

print("전체 텍스트 정제 완료")

In [ ]:
ocr_files = [f for f in os.listdir(ocr_output_dir) if f.endswith(".txt")]
cleaned_files = [f for f in os.listdir(cleaned_output_dir) if f.endswith(".txt")]

print("OCR 결과 txt 개수:", len(ocr_files))
print("정제 결과 txt 개수:", len(cleaned_files))

## 3단계 결과 정리
OCR 테스트 결과, 일반 본문 영역은 비교적 인식되었으나 표 영역에서는 행과 열 구조가 유지되지 않고 숫자 및 문자가 오인식되는 문제가 확인되었다.

이는 페이지 전체를 하나의 이미지로 OCR 처리했기 때문에 발생한 한계로 판단된다.

따라서 이후 단계에서는 문서 구조화 과정을 통해 본문, 표, 이미지, 그래프 영역을 블록 단위로 분리하고, 각 영역에 적합한 방식으로 OCR 및 해석을 수행할 필요가 있다.